## Importando Bibliotecas e Carregando os Dados

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import kurtosis

In [ ]:
caminho_atual = Path(".").resolve()
if caminho_atual.name == "notebooks":
    raiz_projeto = caminho_atual.parent
else:
    raiz_projeto = caminho_atual

pasta_interim = raiz_projeto / "data" / "interim" / "2nd_test" / "2nd_test"
arquivos = sorted([f for f in pasta_interim.iterdir() if f.is_file()])

print(f"Diretório base: {raiz_projeto}")
print(f"{len(arquivos)} arquivos encontrados")

In [ ]:
# Lendo o primeiro arquivo (snapshot de 1 segundo com 20.480 pontos)
sample = pd.read_csv(arquivos[0], sep="\t", header=None)
sample.columns = ["b1", "b2", "b3", "b4"]

print(f"Formato de cada arquivo bruto: {sample.shape[0]} linhas x {sample.shape[1]} colunas")
display(sample.head())

## 1. Análise das Variáveis do Snapshot Bruto (Domínio do Tempo)

In [ ]:
summary_list = []

for col in sample.columns:
    summary_list.append({
        "Feature": col,
        "Tipo de Dado": sample[col].dtype,
        "Valores Totais": sample[col].count(),
        "Valores Nulos": sample[col].isnull().sum(),
        "Proporção de Nulos": f"{sample[col].isnull().mean():.2%}",
        "Mínimo": sample[col].min(),
        "Máximo": sample[col].max(),
        "RMS": np.sqrt(np.mean(sample[col] ** 2)),
        "Exemplos": sample[col].head(3).tolist()
    })

summary_df = pd.DataFrame(summary_list)
display(summary_df)

In [ ]:
# Comparando um arquivo saudável (início) com um arquivo em falha severa (final do teste)
sample_falha = pd.read_csv(arquivos[950], sep="\t", header=None)
sample_falha.columns = ["b1", "b2", "b3", "b4"]

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(14, 4.5), sharey=True)

axes[0].plot(sample["b1"], color="#2980b9", lw=0.6)
axes[0].set_title(f"Rolamento 1 Saudável (Arquivo 0: {arquivos[0].name})")
axes[0].set_xlabel("Amostras (20.000 Hz)")
axes[0].set_ylabel("Aceleração (g)")
axes[0].grid(alpha=0.3)

axes[1].plot(sample_falha["b1"], color="#c0392b", lw=0.6)
axes[1].set_title(f"Rolamento 1 com Falha na Pista Externa (Arquivo 950: {arquivos[950].name})")
axes[1].set_xlabel("Amostras (20.000 Hz)")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Evolução Histórica das Features ao Longo da Vida dos Rolamentos

In [ ]:
# Carregando a tabela consolidada com as features extraídas dos 984 arquivos
caminho_features = raiz_projeto / "data" / "processed" / "2nd_test_features.csv"
df = pd.read_csv(caminho_features)

print(f"Dataset de Features: {df.shape[0]} arquivos x {df.shape[1]} colunas")
display(df.head())

In [ ]:
# Estatísticas descritivas das features do Rolamento 1 (b1)
colunas_b1 = [c for c in df.columns if c.startswith("b1_")]
print("Estatística Descritiva das Features do Rolamento 1 (b1):")
display(df[colunas_b1].describe())

## 3. Comparação da Energia de Vibração (RMS e Curtose) nos 4 Rolamentos

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(15, 5))

# 1. Evolução do RMS nos 4 rolamentos
for b, cor in zip(["b1", "b2", "b3", "b4"], ["#c0392b", "#2980b9", "#27ae60", "#8e44ad"]):
    axes[0].plot(df["idx"], df[f"{b}_rms"], label=f"Rolamento {b.upper()}", lw=1.5, color=cor)

axes[0].axvline(532, color="black", linestyle="--", label="Início da Falha (Idx 532)")
axes[0].set_title("Evolução do RMS ao Longo do Tempo")
axes[0].set_xlabel("Índice do Arquivo (1 arquivo a cada 10 min)")
axes[0].set_ylabel("RMS (g)")
axes[0].legend()
axes[0].grid(alpha=0.3)

# 2. Evolução da Curtose nos 4 rolamentos
for b, cor in zip(["b1", "b2", "b3", "b4"], ["#c0392b", "#2980b9", "#27ae60", "#8e44ad"]):
    axes[1].plot(df["idx"], df[f"{b}_kurtosis"], label=f"Rolamento {b.upper()}", lw=1.5, color=cor)

axes[1].axvline(532, color="black", linestyle="--", label="Início da Falha (Idx 532)")
axes[1].set_title("Evolução da Curtose ao Longo do Tempo")
axes[1].set_xlabel("Índice do Arquivo (1 arquivo a cada 10 min)")
axes[1].set_ylabel("Curtose")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

- O **Rolamento 1 (`b1`)** é o que apresenta degradação severa na pista externa a partir do arquivo `532`, enquanto `b2`, `b3` e `b4` permanecem saudáveis até os momentos finais.
- Nos arquivos `982` e `983` o motor foi desligado (`RMS ≈ 0`).

## 4. Analisando a Distribuição das Features do Rolamento 1 (`b1`)

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(15, 8.5))

features_plot = [
    ("b1_rms", "RMS (Energia Global)"),
    ("b1_kurtosis", "Curtose (Impulsividade)"),
    ("b1_crest_factor", "Fator de Crista"),
    ("b1_peak", "Valor de Pico"),
    ("b1_energy_bpfo", "Energia na Banda BPFO (~236 Hz)"),
    ("b1_energy_high", "Energia em Alta Frequência (2k-5k Hz)")
]

for ax, (col, titulo) in zip(axes.flatten(), features_plot):
    df[col].plot(kind="hist", bins=30, ax=ax, title=titulo, color="#2980b9", edgecolor="black", alpha=0.8)
    ax.set_xlabel(col)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Investigação da Correlação entre as Características do Rolamento 1 (`b1`)

In [ ]:
# Criando uma coluna binária apenas para analisar a correlação com o período de falha (idx >= 532)
df_corr = df[df["idx"] <= 981].copy()
df_corr["Estado_Falha"] = (df_corr["idx"] >= 532).astype(int)

matriz_correlacao = df_corr[colunas_b1 + ["Estado_Falha"]].corr()

corr_alvo = matriz_correlacao["Estado_Falha"].drop("Estado_Falha").sort_values(ascending=False)
print("Correlação das Features do Rolamento 1 com o Estado de Falha:")
display(corr_alvo)

plt.figure(figsize=(10, 8))
sns.heatmap(
    matriz_correlacao,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1, vmax=1,
    linewidths=0.5
)
plt.title("Matriz de Correlação — Rolamento 1 (b1)")
plt.tight_layout()
plt.show()

## 6. Validação Física via Espectro de Frequência (FFT e BPFO ~ 236,4 Hz)

In [ ]:
# Comparando a FFT do Rolamento 1: Saudável (Idx 100) vs. Início de Falha (Idx 600) vs. Falha Severa (Idx 950)
taxa_amostragem = 20000
bpfo = 236.4

plt.figure(figsize=(12, 5))

for idx_arq, rotulo, cor in [(100, "Saudável (Idx 100)", "#27ae60"), (600, "Degradação Inicial (Idx 600)", "#e67e22"), (950, "Falha Severa (Idx 950)", "#c0392b")]:
    sinal = pd.read_csv(arquivos[idx_arq], sep="\t", header=None)[0].values
    sinal = sinal - np.mean(sinal)
    
    freqs = np.fft.rfftfreq(len(sinal), d=1.0 / taxa_amostragem)
    fft_mag = np.abs(np.fft.rfft(sinal)) / len(sinal)
    
    mascara = freqs <= 1000
    plt.plot(freqs[mascara], fft_mag[mascara], label=rotulo, color=cor, lw=1.2, alpha=0.85)

# Destacando a frequência teórica de falha na pista externa (BPFO) e seus harmônicos
for h in [1, 2, 3]:
    plt.axvline(bpfo * h, color="black", linestyle="--", alpha=0.6, label=f"{h}x BPFO ({bpfo*h:.1f} Hz)" if h == 1 else f"{h}x BPFO")

plt.title("Espectro de Frequência (FFT) — Verificação Física da Falha na Pista Externa (BPFO)")
plt.xlabel("Frequência (Hz)")
plt.ylabel("Magnitude")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()